In [ ]:
import random
import os
import numpy as np
import time
import torch
import torch.nn as nn 
import pandas as pd 
import argparse
import torch.optim as optim
import torch.utils.data as data
from tensorboardX import SummaryWriter
import scipy.sparse as sp
import torch.optim as optim
import torch.backends.cudnn as cudnn

参数设置：

In [ ]:
#设置伪随机数，方便实验结果复现
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # 即使没有 CUDA，也可以设置，不会产生影响
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # 在没有 CUDA 的情况下通常将其设置为 False

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--seed", type=int, default=42,help="Seed")
parser.add_argument("--lr", type=float, default=0.01, help="learning rate")
parser.add_argument("--lamda", type=float, default=0.001, help="model regularization rate")
parser.add_argument("--batch_size", type=int, default=4096, help="batch size for training")
parser.add_argument("--epochs", type=int,default=35,  help="training epoches")
parser.add_argument("--top_k", type=int, default=10, help="compute metrics@top_k")
parser.add_argument("--factor_num", type=int,default=32, help="predictive factors numbers in the model")
parser.add_argument("--num_ng", type=int,default=4, help="sample negative items for training")
parser.add_argument("--test_num_ng", type=int,default=100, help="sample part of negative items for testing")
parser.add_argument("--out", default=True,help="save model or not")
parser.add_argument("--gpu", type=str,default="0",  help="gpu card ID")
args=parser.parse_known_args()[0]

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
cudnn.benchmark = True

seed_everything(args.seed)

处理数据：

In [ ]:
class BPRData(data.Dataset):
    def __init__(self, features):
        super(BPRData, self).__init__()
        self.features = pd.read_csv(features,sep=',', names=['user_id', 'item_i', 'item_j'], engine='python')

    def __len__(self):
        return  len(self.features)
    
    def __getitem__(self, idx):
        features = self.features
        
        user = features.iloc[idx, 0]
        item_i = features.iloc[idx, 1]
        item_j = features.iloc[idx, 2] 
        
        return user, item_i, item_j 

In [ ]:
class BPR(nn.Module):
    def __init__(self, user_num, item_num, factor_num):
        super(BPR, self).__init__()
        """
        user_num: number of users;
        item_num: number of items;
        factor_num: number of predictive factors.
        """
        self.embed_user = nn.Embedding(user_num, factor_num)
        self.embed_item = nn.Embedding(item_num, factor_num)

        nn.init.normal_(self.embed_user.weight, std=0.01)
        nn.init.normal_(self.embed_item.weight, std=0.01)

    def forward(self, user, item_i, item_j):
        user = self.embed_user(user)
        item_i = self.embed_item(item_i)
        item_j = self.embed_item(item_j)

        prediction_i = (user * item_i).sum(dim=-1)
        prediction_j = (user * item_j).sum(dim=-1)
        return prediction_i, prediction_j

评估：

In [ ]:
def hit(ng_item, pred_items):
    if ng_item in pred_items:
        return 1
    return 0

def ndcg(ng_item, pred_items):
    if ng_item in pred_items:
        index = pred_items.index(ng_item)
        return np.reciprocal(np.log2(index+2))
    return 0

In [ ]:
n=3
HRs, NDCGs = [], []
for part in range(1,n+1):
    main_path = f'./data/BPR_div_shard{n}/'
    test_data = main_path + f'sub_test{part}.csv' 
    test_dataset = BPRData(test_data)
    test_loader = data.DataLoader(test_dataset,
            batch_size=args.test_num_ng+1, shuffle=False, num_workers=0)
    best_hr = 0
    start_time = time.time()
    user_num = 6040
    item_num = 3706

    # set model and optimizer
    model = BPR(user_num, item_num, args.factor_num)
    model = model.cuda()
    model.eval()
    model = torch.load(f'./models1/model_div{n}/div_shard{part}.pth')
    
#     state_dict = torch.load(f'./models/model_div{n}/div_shard{part}.pth')
#     model.load_state_dict(state_dict)
    for user, item_i, item_j in test_loader:
        user = user.cuda()
        item_i = item_i.cuda()
        item_j = item_j.cuda() # not useful when testing

        prediction_i, prediction_j = model(user, item_i, item_j)
        _, indices = torch.topk(prediction_i, args.top_k)
        recommends = torch.take(
                item_i, indices).cpu().numpy().tolist()

        gt_item = item_i[0].item()
        HRs.append(hit(gt_item, recommends))
        NDCGs.append(ndcg(gt_item, recommends))
        
HR = np.mean(HRs)
NDCG = np.mean(NDCGs)

print("HR: {:.5f}\tNDCG: {:.5f}".format(np.mean(HR), np.mean(NDCG)))

if HR > best_hr:
    elapsed_time = time.time() - start_time#到最好HR的epoch花费的时间
    best_hr, best_ndcg = HR, NDCG

# writer.close()